In [ ]:
## folder create
mkdir -p ~/titanic_mapreduce
cd ~/titanic_mapreduce

याचा अर्थ, तुम्ही /home/admin1/titanic_mapreduce नावाच्या फोल्डरमध्ये काम करत आहात.
आता फक्त तुमची titanic.csv फाइल त्या फोल्डरमध्ये ठेवा.

head -n 5 titanic.csv

gedit mapper_avg_age_male.py
(import sys, csv

reader = csv.DictReader(sys.stdin)
for row in reader:
    try:
        survived = row.get('Survived', '').strip()
        sex = row.get('Sex', '').strip().lower()
        age = row.get('Age', '').strip()
    except Exception:
        continue

    if survived == '0' and sex == 'male' and age not in ('', 'NA', 'None'):
        print(f"male\t{age}"))

chmod +x mapper_avg_age_male.py

gedit reducer_avg_age_male.py
#!/usr/bin/env python3
import sys

total_age = 0.0
count = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    parts = line.split('\t')
    if len(parts) != 2:
        continue
    key, val = parts
    try:
        age = float(val)
    except:
        continue
    total_age += age
    count += 1

if count > 0:
    avg = total_age / count
    print(f"average_age_deceased_males\t{avg:.2f}")
else:
    print("average_age_deceased_males\tNA")

chmod +x reducer_avg_age_male.py

cat titanic.csv | python3 mapper_avg_age_male.py | sort | python3 reducer_avg_age_male.py

gedit mapper_female_class.py 
#!/usr/bin/env python3
import sys, csv

reader = csv.DictReader(sys.stdin)
for row in reader:
    try:
        survived = row.get('Survived', '').strip()
        sex = row.get('Sex', '').strip().lower()
        pclass = row.get('Pclass', '').strip()
    except Exception:
        continue

    if survived == '0' and sex == 'female' and pclass:
        print(f"{pclass}\t1")

chmod +x mapper_female_class.py

gedit reducer_female_class.py 
#!/usr/bin/env python3
import sys

counts = {}

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    parts = line.split('\t')
    if len(parts) != 2:
        continue
    cls, val = parts
    try:
        counts[cls] = counts.get(cls, 0) + int(val)
    except:
        continue

for cls in sorted(counts.keys(), key=lambda x: int(x)):
    print(f"class_{cls}\t{counts[cls]}")

chmod +x reducer_female_class.py

cat titanic.csv | python3 mapper_female_class.py | sort | python3 reducer_female_class.py

## Hadoop वर रन करण्यासाठी
hdfs dfs -mkdir -p /titanic_input
hdfs dfs -put -f titanic.csv /titanic_input/

hdfs dfs -rm -r -f /titanic_output_males

hadoop jar /home/admin1/hadoop-3.3.1/share/hadoop/tools/lib/hadoop-streaming-3.3.1.jar \
  -file mapper_avg_age_male.py -mapper mapper_avg_age_male.py \
  -file reducer_avg_age_male.py -reducer reducer_avg_age_male.py \
  -input /titanic_input/titanic.csv -output /titanic_output_males

hdfs dfs -cat /titanic_output_males/part-00000


hdfs dfs -rm -r -f /titanic_output_female_class

hadoop jar /home/admin1/hadoop-3.3.1/share/hadoop/tools/lib/hadoop-streaming-3.3.1.jar \
  -file mapper_female_class.py -mapper mapper_female_class.py \
  -file reducer_female_class.py -reducer reducer_female_class.py \
  -input /titanic_input/titanic.csv -output /titanic_output_female_class

hdfs dfs -cat /titanic_output_female_class/part-00000



